# Day 072 — Exercise 4: search_transcript

**What you'll build:** `search_transcript(result, query, case_sensitive=False) -> list[dict]` — find segments containing a query string, enabling jump-to-moment navigation.

**Why it matters:** Segment-level search is the core of podcast indexing, meeting search, and lecture navigation. Finding the exact timestamp when a topic was mentioned is not possible with plain text transcription.

In [ ]:
import os, tempfile

def _format_time(seconds):
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return f'{h:02d}:{m:02d}:{s:02d}'

def format_transcript(result, include_timestamps=False):
    if not include_timestamps:
        return result.get('text', '').strip()
    lines = []
    for seg in result.get('segments', []):
        ts = _format_time(seg.get('start', 0.0))
        lines.append(f'[{ts}] {seg.get("text", "").strip()}')
    return '\n'.join(lines)

def extract_segments(result):
    out = []
    for seg in result.get('segments', []):
        logprob = seg.get('avg_logprob', -1.0)
        confidence = min(1.0, max(0.0, 1.0 + logprob))
        out.append({'start': float(seg.get('start', 0.0)),
                    'end':   float(seg.get('end', 0.0)),
                    'text':  seg.get('text', '').strip(),
                    'confidence': round(confidence, 4)})
    return out

def transcribe_audio(source, transcribe_fn=None, model='base'):
    if transcribe_fn is not None:
        return transcribe_fn(source)
    import whisper as _whisper
    mdl = _whisper.load_model(model)
    if isinstance(source, (bytes, bytearray)):
        with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f:
            f.write(source); tmp = f.name
        try:
            return mdl.transcribe(tmp)
        finally:
            os.unlink(tmp)
    return mdl.transcribe(str(source))

def search_transcript(result, query, case_sensitive=False):
    segments = extract_segments(result)
    if not case_sensitive:
        q = query.lower()
        return [s for s in segments if q in s['text'].lower()]
    return [s for s in segments if query in s['text']]

_MOCK_RESULT = {
    'text': ' Hello world. This is a test of speech recognition.',
    'language': 'en',
    'segments': [
        {'id': 0, 'start': 0.0, 'end': 3.2, 'text': ' Hello world.',
         'avg_logprob': -0.25, 'no_speech_prob': 0.01},
        {'id': 1, 'start': 3.2, 'end': 7.8,
         'text': ' This is a test of speech recognition.',
         'avg_logprob': -0.30, 'no_speech_prob': 0.02},
    ],
}
_mock_transcribe = lambda source: _MOCK_RESULT


## Task

Implement `search_transcript`:

1. `segments = extract_segments(result)`
2. If `not case_sensitive`: `q = query.lower()`, filter `[s for s in segments if q in s['text'].lower()]`
3. Else: filter `[s for s in segments if query in s['text']]`

## Your Implementation

In [ ]:
def search_transcript(result: dict, query: str,
                      case_sensitive: bool = False) -> list:
    """Find segments whose text contains the query string.

    Args:
        result:         Whisper result dict
        query:          Search string
        case_sensitive: Default False (case-insensitive comparison)
    Returns:
        list of matching segment dicts in extract_segments format
    """
    raise NotImplementedError


In [ ]:
def search_transcript(result, query, case_sensitive=False):
    segments = extract_segments(result)
    if not case_sensitive:
        q = query.lower()
        return [s for s in segments if q in s['text'].lower()]
    return [s for s in segments if query in s['text']]


## Automated checks

In [ ]:

score, total = 0, 5
try:
    # case-insensitive hit
    hits = search_transcript(_MOCK_RESULT, 'hello')
    assert len(hits) == 1 and hits[0]['text'] == 'Hello world.'
    score += 1; print("✅ case-insensitive search finds 'hello' in 'Hello world.'")

    # case-insensitive miss
    no_hits = search_transcript(_MOCK_RESULT, 'elephant')
    assert no_hits == []
    score += 1; print("✅ no match returns empty list")

    # case-sensitive hit
    cs_hits = search_transcript(_MOCK_RESULT, 'Hello', case_sensitive=True)
    assert len(cs_hits) == 1
    score += 1; print("✅ case-sensitive match works")

    # case-sensitive miss (wrong case)
    cs_miss = search_transcript(_MOCK_RESULT, 'hello', case_sensitive=True)
    assert cs_miss == [], f"Expected [], got {cs_miss}"
    score += 1; print("✅ case-sensitive: wrong case produces no match")

    # result has correct format (from extract_segments)
    hits2 = search_transcript(_MOCK_RESULT, 'test')
    assert len(hits2) == 1
    assert all(k in hits2[0] for k in ('start', 'end', 'text', 'confidence'))
    assert hits2[0]['start'] == 3.2
    score += 1; print("✅ results in extract_segments format with correct timestamps")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def search_transcript(result, query, case_sensitive=False):
    segments = extract_segments(result)
    if not case_sensitive:
        q = query.lower()
        return [s for s in segments if q in s['text'].lower()]
    return [s for s in segments if query in s['text']]
```

**Why call `extract_segments` rather than filtering `result['segments']`?** Two reasons: (1) the returned dicts have consistent, clean keys (stripped text, float start/end, confidence); (2) if `extract_segments` logic changes later, `search_transcript` inherits the change automatically.

</details>